In [16]:
from pyspark.sql import SparkSession, functions as F
from pyspark.sql.types import StructType, StructField, StringType, DoubleType, IntegerType
import numpy as np
import pandas as pd
from scipy.spatial import cKDTree

In [17]:



spark = (
    SparkSession.builder.appName("Rental Price Prediction - Proximity Features")
    .config("spark.sql.repl.eagerEval.enabled", True)
    .config("spark.sql.parquet.cacheMetadata", "true")
    .config("spark.sql.session.timeZone", "Etc/UTC")
    .config("spark.driver.memory", "8g")
    .config("spark.executor.memory", "4g")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("ERROR")

# --- Schools schema, full column set (matches dv402-SchoolLocations2025.csv) ---
schools_schema = StructType([
    StructField("Education_Sector",      StringType(), True),
    StructField("Entity_Type",           StringType(), True),
    StructField("School_No",             StringType(), True),
    StructField("School_Name",           StringType(), True),
    StructField("School_Type",           StringType(), True),
    StructField("School_Status",         StringType(), True),
    StructField("Address_Line_1",        StringType(), True),
    StructField("Address_Line_2",        StringType(), True),
    StructField("Address_Town",          StringType(), True),
    StructField("Address_State",         StringType(), True),
    StructField("Address_Postcode",      IntegerType(), True),
    StructField("Postal_Address_Line_1", StringType(), True),
    StructField("Postal_Address_Line_2", StringType(), True),
    StructField("Postal_Town",           StringType(), True),
    StructField("Postal_State",          StringType(), True),
    StructField("Postal_Postcode",       IntegerType(), True),
    StructField("Full_Phone_No",         StringType(), True),
    StructField("Region",                StringType(), True),
    StructField("Area",                  StringType(), True),
    StructField("LGA_ID",                IntegerType(), True),
    StructField("LGA_Name",              StringType(), True),
    StructField("LGA_TYPE",              StringType(), True),
    StructField("X",                     DoubleType(), True),  # longitude
    StructField("Y",                     DoubleType(), True),  # latitude
])

In [10]:
SCHOOL_DATA_PATH = "../rea_data/landing/domain/Data/SchoolLocations2025.csv"
df_schools = spark.read.csv(SCHOOL_DATA_PATH, header=True, schema=schools_schema)

In [11]:
df_schools.printSchema()

root
 |-- Education_Sector: string (nullable = true)
 |-- Entity_Type: string (nullable = true)
 |-- School_No: string (nullable = true)
 |-- School_Name: string (nullable = true)
 |-- School_Type: string (nullable = true)
 |-- School_Status: string (nullable = true)
 |-- Address_Line_1: string (nullable = true)
 |-- Address_Line_2: string (nullable = true)
 |-- Address_Town: string (nullable = true)
 |-- Address_State: string (nullable = true)
 |-- Address_Postcode: integer (nullable = true)
 |-- Postal_Address_Line_1: string (nullable = true)
 |-- Postal_Address_Line_2: string (nullable = true)
 |-- Postal_Town: string (nullable = true)
 |-- Postal_State: string (nullable = true)
 |-- Postal_Postcode: integer (nullable = true)
 |-- Full_Phone_No: string (nullable = true)
 |-- Region: string (nullable = true)
 |-- Area: string (nullable = true)
 |-- LGA_ID: integer (nullable = true)
 |-- LGA_Name: string (nullable = true)
 |-- LGA_TYPE: string (nullable = true)
 |-- X: double (nullabl

In [9]:
df_schools.show(5, truncate=False)

+----------------+-----------+---------+-----------------------------+-----------+-------------+-------------------------+--------------+--------------+-------------+----------------+---------------------+---------------------+--------------+------------+---------------+-------------+----------------------+-----------------------+------+----------------+---------+----------+----------+
|Education_Sector|Entity_Type|School_No|School_Name                  |School_Type|School_Status|Address_Line_1           |Address_Line_2|Address_Town  |Address_State|Address_Postcode|Postal_Address_Line_1|Postal_Address_Line_2|Postal_Town   |Postal_State|Postal_Postcode|Full_Phone_No|Region                |Area                   |LGA_ID|LGA_Name        |LGA_TYPE |X         |Y         |
+----------------+-----------+---------+-----------------------------+-----------+-------------+-------------------------+--------------+--------------+-------------+----------------+---------------------+-----------------

In [12]:

RENTAL_PATH = "../rea_data/curated/vic_rentals_clean.parquet/"
df = spark.read.parquet(RENTAL_PATH)

In [13]:
df.printSchema()

root
 |-- listing_id: long (nullable = true)
 |-- suburb: string (nullable = true)
 |-- postcode: string (nullable = true)
 |-- weekly_rent: double (nullable = true)
 |-- bond: double (nullable = true)
 |-- available_date: string (nullable = true)
 |-- date_listed: date (nullable = true)
 |-- days_listed: double (nullable = true)
 |-- bedrooms: double (nullable = true)
 |-- bathrooms: double (nullable = true)
 |-- carspaces: double (nullable = true)
 |-- property_type: string (nullable = true)
 |-- address: string (nullable = true)
 |-- lat: double (nullable = true)
 |-- lon: double (nullable = true)
 |-- scraped_date: timestamp (nullable = true)
 |-- domain_page_id: string (nullable = true)
 |-- property_id: string (nullable = true)
 |-- photo_count: integer (nullable = true)
 |-- video_count: integer (nullable = true)
 |-- floorplans_count: integer (nullable = true)
 |-- virtual_tour: boolean (nullable = true)
 |-- primary_type: string (nullable = true)
 |-- secondary_type: string (n

In [14]:
df.show(5, truncate=False)

+----------+----------------+--------+-----------+------+--------------------------+-----------+-----------+--------+---------+---------+-----------------------+-------------------+----------+---------+-------------------+--------------+-----------+-----------+-----------+----------------+------------+---------------+-----------------------+--------------------------+---------+-------------------------------------+-----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+--------------------------------------------------------------------------------+---------+-------------+-----------------+-----------------------+------------+---------------+-------------------+---------------------+---------------------+-------------------+--------+----------------+---------+-------------+------------+-----------------+----------+--------

### Create spacial location features

In [18]:

# --- Only keep open schools with valid coordinates ---
df_schools_open = df_schools.filter(F.col("School_Status") == "O")

df_schools_geo_raw = df_schools_open.select(
    "School_Type", F.col("X").alias("school_lon"), F.col("Y").alias("school_lat")
)

df_schools_geo = df_schools_geo_raw.filter(
    F.col("school_lon").isNotNull() & F.col("school_lat").isNotNull()
)

n_before = df_schools_geo_raw.count()
n_after = df_schools_geo.count()
n_dropped = n_before - n_after
print(f"Open schools before null-coordinate filter: {n_before}")
print(f"Dropped for missing lat/lon: {n_dropped}")
print(f"Remaining after filter: {n_after}")
if n_dropped > 0:
    print("Rows dropped:")
    df_schools_geo_raw.filter(
        F.col("school_lon").isNull() | F.col("school_lat").isNull()
    ).show(truncate=False)

schools_pd = df_schools_geo.toPandas()
schools_pd["School_Type"] = schools_pd["School_Type"].str.strip()

# Also flag any null/blank School_Type, since those rows silently match
# none of the feature_groups below (.isin() on NaN is False).
n_null_type = schools_pd["School_Type"].isna().sum()
print(f"Schools with null School_Type (will be excluded from all groups): {n_null_type}")

# Pri/Sec schools offer both levels, so they contribute to both the
# "primary" and "secondary" nearest-distance groups, in addition to
# their own dedicated group.
feature_groups = {
    "primary":   ["Primary", "Pri/Sec"],
    "secondary": ["Secondary", "Pri/Sec"],
    "pri_sec":   ["Pri/Sec"],
    "special":   ["Special"],
    "language":  ["Language"],
}

print("\nFeature groups:")
for name, types in feature_groups.items():
    n = schools_pd["School_Type"].isin(types).sum()
    print(f"  nearest_{name}_school  <-  {types}  ({n} schools)")

Open schools before null-coordinate filter: 2297
Dropped for missing lat/lon: 0
Remaining after filter: 2297


/home/jovinguyen/miniconda3/envs/tf_env/lib/python3.12/site-packages/pyspark/sql/pandas/conversion.py:298: FutureWarning: PySpark does not yet fully support pandas >= 3.0.0. Some features may not work correctly. It is recommended to use pandas < 3.0.0 for now.
  require_minimum_pandas_version()


Schools with null School_Type (will be excluded from all groups): 0

Feature groups:
  nearest_primary_school  <-  ['Primary', 'Pri/Sec']  (1815 schools)
  nearest_secondary_school  <-  ['Secondary', 'Pri/Sec']  (608 schools)
  nearest_pri_sec_school  <-  ['Pri/Sec']  (248 schools)
  nearest_special_school  <-  ['Special']  (117 schools)
  nearest_language_school  <-  ['Language']  (5 schools)


### Prep Rent Data

In [19]:
rentals_geo = (
    df
    .select("listing_id", "lat", "lon")
    .filter(F.col("lat").isNotNull() & F.col("lon").isNotNull())
)

rentals_pd = rentals_geo.toPandas()
print(f"Rentals with valid coordinates: {len(rentals_pd):,}")

Rentals with valid coordinates: 12,713


/home/jovinguyen/miniconda3/envs/tf_env/lib/python3.12/site-packages/pyspark/sql/pandas/conversion.py:298: FutureWarning: PySpark does not yet fully support pandas >= 3.0.0. Some features may not work correctly. It is recommended to use pandas < 3.0.0 for now.
  require_minimum_pandas_version()


### Nearest neighbour distance per feature group

In [20]:
rental_points = rentals_pd[["lon", "lat"]].to_numpy()

distance_cols = {}
for name, types in feature_groups.items():
    type_points = schools_pd.loc[
        schools_pd["School_Type"].isin(types), ["school_lon", "school_lat"]
    ].to_numpy()

    if len(type_points) == 0:
        continue

    tree = cKDTree(type_points)
    nearest_dist, _ = tree.query(rental_points, k=1)  # Euclidean distance, decimal degrees

    distance_cols[f"nearest_{name}_school"] = nearest_dist

nearest_school_pd = pd.DataFrame(distance_cols)
nearest_school_pd.insert(0, "listing_id", rentals_pd["listing_id"].values)
nearest_school_pd.head()

,listing_id,nearest_primary_school,nearest_secondary_school,nearest_pri_sec_school,nearest_special_school,nearest_language_school
0,16782629,0.007081,0.020536,0.030306,0.018259,0.043401
1,17471867,0.008610,0.015167,0.036099,0.018122,0.037726
2,17721851,0.011611,0.020039,0.032525,0.021861,0.042065
3,17725855,0.009289,0.016378,0.034988,0.018594,0.038899
4,17745057,0.008365,0.015273,0.035883,0.017836,0.037902


### KM Unit conversion

In [23]:
KM_PER_DEGREE = 111.0  # rough approximation, valid for this small geographic extent

rental_points = rentals_pd[["lon", "lat"]].to_numpy()

distance_cols = {}
for name, types in feature_groups.items():
    type_points = schools_pd.loc[
        schools_pd["School_Type"].isin(types), ["school_lon", "school_lat"]
    ].to_numpy()

    if len(type_points) == 0:
        continue

    tree = cKDTree(type_points)
    nearest_dist, _ = tree.query(rental_points, k=1)  # Euclidean distance, decimal degrees

    distance_cols[f"nearest_{name}_school_km"] = nearest_dist * KM_PER_DEGREE

nearest_school_pd = pd.DataFrame(distance_cols)
nearest_school_pd.insert(0, "listing_id", rentals_pd["listing_id"].values)
nearest_school_pd.head()

,listing_id,nearest_primary_school_km,nearest_secondary_school_km,nearest_pri_sec_school_km,nearest_special_school_km,nearest_language_school_km
0,16782629,0.786003,2.279515,3.363967,2.026752,4.817560
1,17471867,0.955668,1.683531,4.006978,2.011559,4.187632
2,17721851,1.288852,2.224330,3.610235,2.426517,4.669163
3,17725855,1.031107,1.817975,3.883628,2.063911,4.317818
4,17745057,0.928530,1.695269,3.983055,1.979819,4.207077


In [24]:
school_km_cols = [c for c in nearest_school_pd.columns if c.endswith("_school_km")]

nearest_school_pd["nearest_any_school_km"] = nearest_school_pd[school_km_cols].min(axis=1)

nearest_school_pd.head()

,listing_id,nearest_primary_school_km,nearest_secondary_school_km,nearest_pri_sec_school_km,nearest_special_school_km,nearest_language_school_km,nearest_any_school_km
0,16782629,0.786003,2.279515,3.363967,2.026752,4.817560,0.786003
1,17471867,0.955668,1.683531,4.006978,2.011559,4.187632,0.955668
2,17721851,1.288852,2.224330,3.610235,2.426517,4.669163,1.288852
3,17725855,1.031107,1.817975,3.883628,2.063911,4.317818,1.031107
4,17745057,0.928530,1.695269,3.983055,1.979819,4.207077,0.928530


### Save as a parquet 

In [25]:
OUTPUT_PATH = "../rea_data/curated/features/school_features.parquet"

nearest_school_sdf = spark.createDataFrame(nearest_school_pd)
nearest_school_sdf.write.mode("overwrite").parquet(OUTPUT_PATH)

print(f"Saved {nearest_school_sdf.count()} rows, {len(nearest_school_sdf.columns)} columns to {OUTPUT_PATH}")

/home/jovinguyen/miniconda3/envs/tf_env/lib/python3.12/site-packages/pyspark/sql/pandas/conversion.py:659: FutureWarning: PySpark does not yet fully support pandas >= 3.0.0. Some features may not work correctly. It is recommended to use pandas < 3.0.0 for now.
  require_minimum_pandas_version()
/home/jovinguyen/miniconda3/envs/tf_env/lib/python3.12/site-packages/pyspark/sql/pandas/conversion.py:936: FutureWarning: PySpark does not yet fully support pandas >= 3.0.0. Some features may not work correctly. It is recommended to use pandas < 3.0.0 for now.
  require_minimum_pandas_version()


Saved 12713 rows, 7 columns to ../rea_data/curated/features/school_features.parquet
